# Temat 01. Audyt i profilowanie danych - od surowego pliku do wiarygodnej tabeli

**Kurs:** Przetwarzanie danych i Feature Engineering · studia podyplomowe *Data Science - Analiza danych od podstaw* · Wydział Informatyki i Telekomunikacji PWr
**Prowadząca:** dr inż. Natalia Piórkowska
**Czas bloku:** 2,5 h (Zjazd I, blok 1)
**Wersja pliku:** **PROWADZĄCY** - z rozwiązaniami i notatkami dydaktycznymi

---

### Po tych zajęciach będziesz umieć

1. Ocenić, czy wczytany plik rzeczywiście zawiera to, co obiecuje jego nagłówek.
2. Odróżnić braki jawne (`NaN`) od braków ukrytych pod tekstem typu `"brak"`, `"N/A"`, `"-"`.
3. Napisać własną funkcję profilującą, która w jednej tabeli podsumowuje cały zbiór.
4. Wykryć duplikaty pełne i duplikaty klucza biznesowego - i wiedzieć, czym się różnią.
5. Zapisać reguły poprawności danych jako kod, a nie jako notatkę na kartce.
6. Przygotować *kartę zbioru danych* - krótki dokument, bez którego analiza nie jest odtwarzalna.

### Zbiory danych wykorzystane w tym temacie

- `pracownicy_brudne` - fikcyjny zbiór kadrowy, celowo zabrudzony (nasz główny poligon)
- `titanic` - klasyka, na porównanie "jak wygląda zbiór już uporządkowany"

---

## 0. Przygotowanie środowiska

Uruchom poniższą komórkę. Jeśli zobaczysz numery wersji bibliotek - jesteśmy gotowi.

In [1]:
# --- konfiguracja startowa (uruchom jako pierwszą komórkę) -------------------
# Działa w Google Colab i lokalnie. W Colab podłącza Dysk Google, odnajduje
# katalog kursu i ustawia go jako miejsce pracy - dzięki temu pliki zapisywane
# do katalogu 'wyniki' nie znikają po zamknięciu notatnika.
import sys
import os
from pathlib import Path

NAZWA_KURSU = "kurs_feature_engineering"

try:
    import google.colab  # noqa: F401
    W_COLAB = True
except ImportError:
    W_COLAB = False

if W_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")          # za pierwszym razem Colab poprosi o zgodę
    MOJ_DYSK = Path("/content/drive/MyDrive")

    # katalog kursu: w "Mój dysk" albo o jeden poziom głębiej
    kandydaci = [MOJ_DYSK / NAZWA_KURSU] + sorted(MOJ_DYSK.glob(f"*/{NAZWA_KURSU}"))
    KORZEN = next((k for k in kandydaci if (k / "dane.py").exists()), None)

    if KORZEN is None:
        # pierwsze uruchomienie: rozpakowujemy paczkę wgraną na Dysk
        import zipfile
        paczki = sorted(MOJ_DYSK.glob(f"{NAZWA_KURSU}*.zip"))
        if not paczki:
            raise FileNotFoundError(
                f"Nie znalazłam katalogu ani pliku {NAZWA_KURSU}.zip w 'Mój dysk'. "
                "Wgraj paczkę kursu do głównego katalogu Dysku Google i uruchom komórkę ponownie."
            )
        with zipfile.ZipFile(paczki[0]) as z:
            z.extractall(MOJ_DYSK)
        KORZEN = MOJ_DYSK / NAZWA_KURSU
        print("Rozpakowano paczkę kursu do:", KORZEN)

    os.chdir(KORZEN / "notatniki")
    sys.path.insert(0, str(KORZEN))
else:
    # praca lokalna: moduł dane.py leży w katalogu głównym materiałów
    for kandydat in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        if (kandydat / "dane.py").exists():
            sys.path.insert(0, str(kandydat))
            break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from dane import wczytaj, lista_zbiorow

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (9, 4.5)

RNG = 2026          # jedno ziarno losowości na cały notatnik - dla powtarzalności
np.random.seed(RNG)

print("Gotowe.", "Colab" if W_COLAB else "lokalnie", "| pandas", pd.__version__, "| numpy", np.__version__)


Mounted at /content/drive
Gotowe. Colab | pandas 2.2.3 | numpy 2.1.3


---

## 1. Dlaczego zaczynamy od audytu

Badania nad praktyką pracy analityków od lat powtarzają ten sam wynik: **większość czasu
w projekcie danych pochłania przygotowanie danych, a nie modelowanie.** Szacunki wahają się
od 50% do 80% - różnią się metodologią, ale nie kierunkiem wniosku.

Powód jest prozaiczny. Model uczy się tego, co mu damy. Jeżeli w kolumnie `miasto` ta sama
Warszawa występuje jako `"Warszawa"`, `"warszawa"` i `"  WARSZAWA "`, to model widzi
**trzy różne miasta**. Nie zgłosi błędu. Po prostu nauczy się gorzej, a my nigdy się
nie dowiemy dlaczego.

Stąd zasada, do której będziemy wracać przez cały kurs:

> **Nigdy nie ufaj plikowi, którego nie sprofilowałeś.**

Zaczynamy od zbioru, który wygląda zupełnie niewinnie.

---

## 2. Pierwszy kontakt ze zbiorem

Cztery polecenia, które wykonujemy **zawsze**, zanim cokolwiek policzymy.

In [2]:
df = wczytaj("pracownicy_brudne")

print("Wymiary (wiersze, kolumny):", df.shape)
df.head()

Wymiary (wiersze, kolumny): (1245, 11)


,id_pracownika,miasto,dzial,wyksztalcenie,staz_lat,wiek,liczba_projektow,godziny_msc,wynagrodzenie,odszedl,data_zatrudnienia
0,1485,Katowice,IT,magister,11.2,42,1,199.0,15933.86,0,2017-05-29
1,1947,Wrocław,Sprzedaż,licencjat,1.5,62,4,166.0,brak,0,09/11/2006
2,2045,Warszawa,Logistyka,średnie,9.7,42,3,148.0,11169.96,0,2023-02-13
3,2175,WROCŁAW,Logistyka,magister,2.1,54,8,160.0,10588.61,1,09/02/2024
4,1186,POZNAŃ,IT,magister,2.0,55,4,179.0,NaN,1,2012-01-29


In [3]:
# Typy kolumn i zużycie pamięci - jedno wywołanie, dużo informacji
df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1245 entries, 0 to 1244
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_pracownika      1245 non-null   int64  
 1   miasto             1245 non-null   object 
 2   dzial              1245 non-null   object 
 3   wyksztalcenie      1245 non-null   object 
 4   staz_lat           1135 non-null   float64
 5   wiek               1245 non-null   int64  
 6   liczba_projektow   1245 non-null   int64  
 7   godziny_msc        1213 non-null   object 
 8   wynagrodzenie      1179 non-null   object 
 9   odszedl            1245 non-null   int64  
 10  data_zatrudnienia  1245 non-null   object 
dtypes: float64(1), int64(4), object(6)
memory usage: 497.8 KB


### Zatrzymajmy się na chwilę

Spójrz na typy kolumn. Co najmniej dwie rzeczy powinny zapalić lampkę ostrzegawczą.

- `wynagrodzenie` i `godziny_msc` to wielkości liczbowe, a pandas wczytał je jako **tekst**.
- `data_zatrudnienia` też jest tekstem, mimo że to data.

Gdy pandas wczyta liczbę jako tekst, to zawsze znaczy jedno: **w kolumnie jest co najmniej
jedna wartość, której nie dało się zinterpretować jako liczby.** Naszym zadaniem jest
znaleźć tę wartość, a nie obejść problem rzutowaniem.

In [4]:
# ZADANIE: sprawdź, które wartości w kolumnie 'wynagrodzenie' nie dają się
# zamienić na liczbę. Przydadzą się: pd.to_numeric(..., errors="coerce"),
# .isna() oraz .value_counts(dropna=False)

podejrzane = df.loc[
    pd.to_numeric(df["wynagrodzenie"], errors="coerce").isna(),
    "wynagrodzenie"
]

print("Liczba wartości nieliczbowych:", podejrzane.notna().sum())
print()
print(podejrzane.value_counts(dropna=False).head(12))

Liczba wartości nieliczbowych: 386

wynagrodzenie
NaN          66
-            37
brak         35
9 746,18      2
15 363,76     2
7 972,57      2
9 780,11      2
13 908,21     2
12 905,14     2
7 516,21      2
8 057,55      2
11 653,34     2
Name: count, dtype: int64


> 💡 **Wskazówka:** `pd.to_numeric(seria, errors="coerce")` zamienia na liczbę wszystko, co się da,
> a resztę ustawia na `NaN`. Porównując wynik z oryginałem, dowiesz się, co się "nie dało".

---

## 3. Braki jawne i braki ukryte

`pandas` rozpoznaje jako brak tylko pewien zamknięty zbiór zapisów (`NaN`, pusta komórka,
`NA`, `null`, `NULL`, `N/A`, `n/a` i kilka innych). Wszystko poza tą listą jest dla niego
zwykłym tekstem.

To znaczy, że `df.isna().sum()` - odruchowe pierwsze polecenie każdego analityka -
potrafi **poważnie niedoszacować** skali braków.

In [5]:
braki_jawne = df.isna().sum()
print("Braki widziane przez pandas (isna):")
print(braki_jawne[braki_jawne > 0].to_string())

Braki widziane przez pandas (isna):
staz_lat         110
godziny_msc       32
wynagrodzenie     66


In [6]:
# ZADANIE: uzupełnij funkcję zliczającą braki ukryte pod tekstem.
MARKERY_BRAKU = {"brak", "brak danych", "n/a", "n/d", "-", "?", "", "nan", "null", "bd", "x"}

def policz_braki_ukryte(ramka):
    """Zlicza wartości tekstowe, które w praktyce oznaczają brak danych."""
    wynik = {}
    for kol in ramka.columns:
        # 1) pomiń braki jawne (.dropna()), zamień kolumnę na tekst,
        #    usuń białe znaki, sprowadź do małych liter
        seria = ramka[kol].dropna().astype(str).str.strip().str.lower()
        # 2) policz, ile wartości należy do zbioru MARKERY_BRAKU
        wynik[kol] = seria.isin(MARKERY_BRAKU).sum()
    return pd.Series(wynik)

braki_ukryte = policz_braki_ukryte(df)

porownanie = pd.DataFrame({"jawne_NaN": df.isna().sum(), "ukryte_tekstem": braki_ukryte})
porownanie["razem"] = porownanie.sum(axis=1)
porownanie["procent"] = (100 * porownanie["razem"] / len(df)).round(1)
porownanie[porownanie["razem"] > 0].sort_values("razem", ascending=False)

,jawne_NaN,ukryte_tekstem,razem,procent
wynagrodzenie,66,72,138,11.1
staz_lat,110,0,110,8.8
wyksztalcenie,0,77,77,6.2
godziny_msc,32,30,62,5.0


### Poprawne wczytanie

Znając markery, możemy wczytać plik jeszcze raz - tym razem uczciwie. `read_csv` przyjmuje
parametr `na_values`, który dopisuje nasze markery do listy rozpoznawanych braków.

In [13]:
# ZADANIE: wczytaj plik jeszcze raz, tym razem podając markery braku,
# a następnie napisz funkcję zamieniającą polski zapis liczby na float.

df = wczytaj(
    "pracownicy_brudne",
    na_values=list(MARKERY_BRAKU),         # uzupełnij listę markerów
    keep_default_na=True,
)

def na_liczbe_pl(seria):
    """Zamienia tekst w formacie polskim ("8 908,18") na liczbę."""
    # podpowiedź: .str.replace(...) trzy razy, potem pd.to_numeric(..., errors="coerce")
    tekst = (
        seria.astype("string")
        .str.replace("\u00a0", "", regex=False)  # spacja nierozdzielająca
        .str.replace(" ", "", regex=False)       # zwykła spacja
        .str.replace(",", ".", regex=False)
        #.replace({"nan": None, "None": None}),      # przecinek dziesiętny
    )
    return pd.to_numeric(tekst, errors="coerce")

df["wynagrodzenie"] = na_liczbe_pl(df["wynagrodzenie"])
df["godziny_msc"] = na_liczbe_pl(df["godziny_msc"])

print(df.isna().sum()[lambda s: s > 0].to_string())
print(df[["wynagrodzenie", "godziny_msc"]].describe().round(2))

wyksztalcenie     77
staz_lat         110
godziny_msc       62
wynagrodzenie    138
       wynagrodzenie  godziny_msc
count         1107.0       1183.0
mean        10414.23       168.18
std          2166.71        18.05
min          3816.55        115.0
25%          8811.72        155.0
50%         10286.41        168.0
75%          11704.3        180.0
max         19491.97        217.0


---

## 4. Funkcja profilująca - jedna tabela zamiast dziesięciu poleceń

Zamiast za każdym razem wpisywać `shape`, `dtypes`, `isna`, `nunique` i `describe`,
napiszemy **jedną funkcję**, która zrobi to za nas i zwróci czytelną tabelę.

To będzie nasze narzędzie na cały kurs - warto je mieć zapisane.

In [14]:
# ZADANIE: napisz funkcję profilującą. Szkielet poniżej - uzupełnij brakujące miejsca.

def profil(ramka, maks_przykladow=3):
    """Zwraca tabelę-podsumowanie każdej kolumny ramki danych."""
    n = len(ramka)
    wiersze = []
    for kol in ramka.columns:
        s = ramka[kol]
        niepuste = s.notna().sum()                      # liczba wartości różnych od NaN
        rekord = {
            "kolumna": kol,
            "typ": str(s.dtype),
            "niepuste": niepuste,
            "braki": s.isna().sum(),
            "braki_%": round(100 * s.isna().sum() / n, 1),
            "unikalne": s.nunique(),                # podpowiedź: .nunique()
            "unikalne_%": round(100 * s.nunique() / n, 1),
            "przyklady": ", ".join(map(str, s.dropna().unique()[:maks_przykladow])),
        }
        if pd.api.types.is_numeric_dtype(s) and niepuste > 0:
            rekord["min"] = round(float(s.min()), 2)
            rekord["mediana"] = round(float(s.median()), 2)
            rekord["max"] = round(float(s.max()), 2)
        wiersze.append(rekord)
    return pd.DataFrame(wiersze).set_index("kolumna")


profil(df)

,typ,niepuste,braki,braki_%,unikalne,unikalne_%,przyklady,min,mediana,max
kolumna,,,,,,,,,,
id_pracownika,int64,1245,0,0.0,1215,97.6,"1485, 1947, 2045",1000.00,1606.00,102024.00
miasto,object,1245,0,0.0,19,1.5,"Katowice, Wrocław, Warszawa",NaN,NaN,NaN
dzial,object,1245,0,0.0,8,0.6,"IT, Sprzedaż, Logistyka",NaN,NaN,NaN
wyksztalcenie,object,1168,77,6.2,5,0.4,"magister, licencjat, średnie",NaN,NaN,NaN
staz_lat,float64,1135,110,8.8,182,14.6,"11.2, 1.5, 9.7",-3.00,5.20,120.00
wiek,int64,1245,0,0.0,47,3.8,"42, 62, 54",1.00,43.00,199.00
liczba_projektow,int64,1245,0,0.0,15,1.2,"1, 4, 3",0.00,4.00,99.00
godziny_msc,Float64,1183,62,5.0,96,7.7,"199.0, 166.0, 148.0",115.00,168.00,217.00
wynagrodzenie,Float64,1107,138,11.1,1067,85.7,"15933.86, 11169.96, 10588.61",3816.55,10286.41,19491.97


### Jak czytać tę tabelę

Kolumna `unikalne_%` jest niedocenianym narzędziem diagnostycznym:

| `unikalne_%` | Co to zwykle znaczy | Co z tym zrobić |
|---|---|---|
| ok. 100% | identyfikator albo cecha ciągła | identyfikator **wyrzucamy z modelu** - nie niesie informacji, a idealnie "dopasuje" model |
| 2 | zmienna binarna | sprawdź balans klas |
| 3-20 | kategoria "normalna" | kandydat do kodowania one-hot |
| 20-200 | kategoria o wysokiej liczności | uwaga przy one-hot, wrócimy do tego w temacie 6 |
| 1 | kolumna stała | **wyrzucamy** - zero informacji |

Sprawdźmy to na naszym zbiorze.

In [15]:
p = profil(df)

print("Kolumny wyglądające na identyfikator (unikalne_% > 95):")
print(list(p[p["unikalne_%"] > 95].index))
print()
print("Kolumny stałe (tylko jedna wartość):")
print(list(p[p["unikalne"] <= 1].index))
print()
print("Kolumny kategoryczne (2-20 różnych wartości):")
print(list(p[(p["unikalne"] >= 2) & (p["unikalne"] <= 20)].index))

Kolumny wyglądające na identyfikator (unikalne_% > 95):
['id_pracownika']

Kolumny stałe (tylko jedna wartość):
[]

Kolumny kategoryczne (2-20 różnych wartości):
['miasto', 'dzial', 'wyksztalcenie', 'liczba_projektow', 'odszedl']


---

## 5. Duplikaty - dwa różne zjawiska pod jedną nazwą

Rozróżniamy:

- **duplikat pełny** - dwa identyczne wiersze we wszystkich kolumnach. Zwykle artefakt
  eksportu danych, prawie zawsze do usunięcia.
- **duplikat klucza biznesowego** - dwa wiersze o tym samym identyfikatorze, ale różnej
  treści. To **nie jest** to samo i nie wolno go usuwać automatycznie: może oznaczać
  historię zmian, może oznaczać błąd, a może dwie różne osoby o tym samym numerze.

In [16]:
print("Duplikaty pełne (wszystkie kolumny identyczne):", df.duplicated().sum())
print("Duplikaty identyfikatora 'id_pracownika':     ", df.duplicated(subset=["id_pracownika"]).sum())
print()

# Obejrzyjmy jeden konkretny przypadek
powtorzone_id = df.loc[df.duplicated(subset=["id_pracownika"], keep=False), "id_pracownika"]
przyklad = powtorzone_id.iloc[0]
df[df["id_pracownika"] == przyklad]

Duplikaty pełne (wszystkie kolumny identyczne): 30
Duplikaty identyfikatora 'id_pracownika':      30



,id_pracownika,miasto,dzial,wyksztalcenie,staz_lat,wiek,liczba_projektow,godziny_msc,wynagrodzenie,odszedl,data_zatrudnienia
2,2045,Warszawa,Logistyka,średnie,9.7,42,3,148.0,11169.96,0,2023-02-13
686,2045,Warszawa,Logistyka,średnie,9.7,42,3,148.0,11169.96,0,2023-02-13


In [17]:
# ZADANIE: policz duplikaty pełne i duplikaty samego identyfikatora,
# a następnie usuń wyłącznie duplikaty pełne. Zapisz, ile wierszy ubyło.

print("Duplikaty pełne:", df.duplicated().sum())
print("Duplikaty id_pracownika:", df.duplicated(subset=["id_pracownika"]).sum())

przed = len(df)
df =  df.drop_duplicates()
print(f"Usunięto {przed - len(df)} wierszy.")

Duplikaty pełne: 30
Duplikaty id_pracownika: 30
Usunięto 30 wierszy.


---

## 6. Reguły poprawności zapisane jako kod

Każdy zbiór ma ograniczenia wynikające ze zdrowego rozsądku i z dziedziny: wiek pracownika
mieści się w przedziale mniej więcej 18-70 lat, staż nie może być dłuższy niż wiek minus 18,
wynagrodzenie jest dodatnie.

Warto te reguły **zapisać jako kod**. Wtedy przy kolejnym pliku wystarczy uruchomić funkcję,
zamiast oglądać wszystko od nowa.

In [18]:
# ZADANIE: uzupełnij funkcję sprawdzającą reguły oraz dopisz własne reguły do słownika.

def sprawdz_reguly(ramka, reguly):
    raport = []
    for opis, warunek in reguly.items():
        maska_ok = warunek(ramka)
        naruszenia =  maska_ok.eq(False).sum()         # policz wiersze, które regułę łamią (pomiń NaN!)
        raport.append({
            "regula": opis,
            "naruszenia": int(naruszenia),
            "procent": round(100 * naruszenia / len(ramka), 2),
        })
    return pd.DataFrame(raport).sort_values("naruszenia", ascending=False)


REGULY = {
    "wiek w przedziale 18-70": lambda d: d["wiek"].between(18, 70),
    "staz nieujemny":          lambda d: d["staz_lat"] >= 0,
    "wynagrodzenie dodatnie":  lambda d: d["wynagrodzenie"] > 0,
    "liczba projektow nieujemna": lambda d: d["liczba_projektow"] >= 0,
    "odejscie ma wartosc 0 lub 1": lambda d: d["odszedl"].isin([0, 1]),
}

sprawdz_reguly(df, REGULY)

,regula,naruszenia,procent
1,staz nieujemny,110,9.05
0,wiek w przedziale 18-70,8,0.66
2,wynagrodzenie dodatnie,0,0.00
3,liczba projektow nieujemna,0,0.00
4,odejscie ma wartosc 0 lub 1,0,0.00


In [19]:
# Obejrzyjmy wiersze, które łamią regułę wieku
df[~df["wiek"].between(18, 70)][["id_pracownika", "wiek", "staz_lat", "dzial"]]

,id_pracownika,wiek,staz_lat,dzial
250,1529,3,0.9,Logistyka
509,1803,3,9.2,IT
891,2174,3,0.9,Logistyka
925,1106,199,4.2,HR
1015,1878,1,18.7,Logistyka
1027,1956,150,1.6,Sprzedaż
1200,1984,150,4.7,IT
1211,1481,199,5.3,IT


---

## 7. Karta zbioru danych (*datasheet*) i wątek prawny

Ostatni element audytu nie jest kodem. Jest dokumentem.

Karta zbioru danych odpowiada na pytania, których nie da się odczytać z pliku CSV:
skąd te dane pochodzą, kto je zebrał, na jakiej podstawie prawnej, do czego wolno
ich użyć, kiedy wygasają. Koncepcję spopularyzował artykuł *Datasheets for Datasets*
(Gebru i in., 2018), dziś jest standardem w dokumentacji projektów ML.

**Dlaczego to jest na tych zajęciach, a nie na wykładzie z prawa?**

Bo decyzje techniczne, które zaraz będziemy podejmować, mają konsekwencje prawne.
Nasz zbiór `pracownicy_brudne` zawiera dane o wynagrodzeniach i stażu konkretnych osób -
w świecie rzeczywistym byłyby to dane osobowe w rozumieniu RODO. Trzy praktyczne wnioski:

1. **Minimalizacja danych** (art. 5 ust. 1 lit. c RODO) - jeżeli identyfikator pracownika
   nie jest potrzebny do modelu, nie powinien w nim być. Usuwając `id_pracownika`,
   robimy jednocześnie dobrą inżynierię cech i dobrą praktykę prawną.
2. **Cechy zastępcze (*proxy*)** - usunięcie kolumny `płeć` nie sprawia, że model
   przestaje różnicować po płci, jeśli zostawimy cechę silnie z nią skorelowaną.
   Do tego wrócimy przy selekcji cech.
3. **Odtwarzalność** - jeśli nie zapiszemy, którą wersję pliku analizowaliśmy i jakie
   decyzje czyszczące podjęliśmy, wynik jest niesprawdzalny.

Poniżej generujemy szkielet takiej karty automatycznie z tego, co już wiemy o zbiorze.

In [20]:
from datetime import date

def karta_zbioru(ramka, nazwa, zrodlo, opis, uwagi_prawne=""):
    """Generuje szkielet karty zbioru danych w formacie Markdown."""
    p = profil(ramka)
    liczbowe = [k for k in ramka.columns if pd.api.types.is_numeric_dtype(ramka[k])]
    tekstowe = [k for k in ramka.columns if k not in liczbowe]
    return f"""
# Karta zbioru danych: {nazwa}

| Pole | Wartość |
|---|---|
| Data sporządzenia | {date.today().isoformat()} |
| Źródło | {zrodlo} |
| Liczba wierszy | {len(ramka)} |
| Liczba kolumn | {ramka.shape[1]} |
| Kolumny liczbowe | {len(liczbowe)} |
| Kolumny nieliczbowe | {len(tekstowe)} |
| Łączny odsetek braków | {round(100 * ramka.isna().sum().sum() / ramka.size, 2)}% |
| Kolumny z brakami | {', '.join(p[p['braki'] > 0].index) or 'brak'} |
| Duplikaty pełne | {int(ramka.duplicated().sum())} |

## Opis
{opis}

## Uwagi prawne i etyczne
{uwagi_prawne or '_do uzupełnienia_'}

## Znane ograniczenia
- _do uzupełnienia po analizie_
"""


print(karta_zbioru(
    df,
    nazwa="pracownicy_brudne",
    zrodlo="Zbiór syntetyczny, wygenerowany na potrzeby kursu (SEED=2026). Dane fikcyjne.",
    opis="Fikcyjne dane kadrowe: dział, miasto, wykształcenie, staż, wynagrodzenie, "
         "fakt odejścia z firmy. Zbiór celowo zawiera typowe defekty danych rzeczywistych.",
    uwagi_prawne="Dane są w pełni syntetyczne - nie dotyczą żadnych realnych osób. "
                 "W odpowiedniku rzeczywistym kolumny 'wynagrodzenie' i 'id_pracownika' "
                 "stanowiłyby dane osobowe; przed analizą wymagana byłaby pseudonimizacja "
                 "i ocena podstawy prawnej przetwarzania.",
))


# Karta zbioru danych: pracownicy_brudne

| Pole | Wartość |
|---|---|
| Data sporządzenia | 2026-09-26 |
| Źródło | Zbiór syntetyczny, wygenerowany na potrzeby kursu (SEED=2026). Dane fikcyjne. |
| Liczba wierszy | 1215 |
| Liczba kolumn | 11 |
| Kolumny liczbowe | 7 |
| Kolumny nieliczbowe | 4 |
| Łączny odsetek braków | 2.82% |
| Kolumny z brakami | wyksztalcenie, staz_lat, godziny_msc, wynagrodzenie |
| Duplikaty pełne | 0 |

## Opis
Fikcyjne dane kadrowe: dział, miasto, wykształcenie, staż, wynagrodzenie, fakt odejścia z firmy. Zbiór celowo zawiera typowe defekty danych rzeczywistych.

## Uwagi prawne i etyczne
Dane są w pełni syntetyczne - nie dotyczą żadnych realnych osób. W odpowiedniku rzeczywistym kolumny 'wynagrodzenie' i 'id_pracownika' stanowiłyby dane osobowe; przed analizą wymagana byłaby pseudonimizacja i ocena podstawy prawnej przetwarzania.

## Znane ograniczenia
- _do uzupełnienia po analizie_



In [21]:
def karta_zbioru(ramka, nazwa, zrodlo, opis, uwagi_prawne=""):
    """Generuje szkielet karty zbioru danych w formacie Markdown."""
    p = profil(ramka)
    liczbowe = [k for k in ramka.columns if pd.api.types.is_numeric_dtype(ramka[k])]
    tekstowe = [k for k in ramka.columns if k not in liczbowe]
    return f"""
# Karta zbioru danych: {nazwa}

| Pole | Wartość |
|---|---|
| Data sporządzenia | {date.today().isoformat()} |
| Źródło | {zrodlo} |
| Liczba wierszy | {len(ramka)} |
| Liczba kolumn | {ramka.shape[1]} |
| Kolumny liczbowe | {len(liczbowe)} |
| Kolumny nieliczbowe | {len(tekstowe)} |
| Łączny odsetek braków | {round(100 * ramka.isna().sum().sum() / ramka.size, 2)}% |
| Kolumny z brakami | {', '.join(p[p['braki'] > 0].index) or 'brak'} |
| Duplikaty pełne | {int(ramka.duplicated().sum())} |

## Opis
{opis}

## Uwagi prawne i etyczne
{uwagi_prawne or '_do uzupełnienia_'}

## Znane ograniczenia
- _do uzupełnienia po analizie_
"""


print(karta_zbioru(
    df,
    nazwa="pracownicy_brudne",
    zrodlo="Zbiór syntetyczny, wygenerowany na potrzeby kursu (SEED=2026). Dane fikcyjne.",
    opis="Fikcyjne dane kadrowe: dział, miasto, wykształcenie, staż, wynagrodzenie, "
         "fakt odejścia z firmy. Zbiór celowo zawiera typowe defekty danych rzeczywistych.",
    uwagi_prawne="Dane są w pełni syntetyczne - nie dotyczą żadnych realnych osób. "
                 "W odpowiedniku rzeczywistym kolumny 'wynagrodzenie' i 'id_pracownika' "
                 "stanowiłyby dane osobowe; przed analizą wymagana byłaby pseudonimizacja "
                 "i ocena podstawy prawnej przetwarzania.",
))


# Karta zbioru danych: pracownicy_brudne
 
| Pole | Wartość |
|---|---|
| Data sporządzenia | 2026-09-26 |
| Źródło | Zbiór syntetyczny, wygenerowany na potrzeby kursu (SEED=2026). Dane fikcyjne. |
| Liczba wierszy | 1215 |
| Liczba kolumn | 11 |
| Kolumny liczbowe | 7 |
| Kolumny nieliczbowe | 4 |
| Łączny odsetek braków | 2.82% |
| Kolumny z brakami | wyksztalcenie, staz_lat, godziny_msc, wynagrodzenie |
| Duplikaty pełne | 0 |
 
## Opis
Fikcyjne dane kadrowe: dział, miasto, wykształcenie, staż, wynagrodzenie, fakt odejścia z firmy. Zbiór celowo zawiera typowe defekty danych rzeczywistych.
 
## Uwagi prawne i etyczne
Dane są w pełni syntetyczne - nie dotyczą żadnych realnych osób. W odpowiedniku rzeczywistym kolumny 'wynagrodzenie' i 'id_pracownika' stanowiłyby dane osobowe; przed analizą wymagana byłaby pseudonimizacja i ocena podstawy prawnej przetwarzania.
 
## Znane ograniczenia
- _do uzupełnienia po analizie_



---

## 8. Podsumowanie - lista kontrolna audytu

Zapisz ją sobie. Będziemy z niej korzystać przy każdym kolejnym zbiorze.

```
[ ]  1. Wymiary zbioru - czy zgadzają się z oczekiwaniem?
[ ]  2. Typy kolumn - czy liczby są liczbami, a daty datami?
[ ]  3. Wartości nieliczbowe w kolumnach liczbowych - dlaczego tam są?
[ ]  4. Braki jawne (NaN) - ile i gdzie?
[ ]  5. Braki ukryte (markery tekstowe) - ile i gdzie?
[ ]  6. Duplikaty pełne - ile?
[ ]  7. Duplikaty klucza biznesowego - ile i co oznaczają?
[ ]  8. Kolumny stałe i identyfikatory - do usunięcia z modelu
[ ]  9. Reguły dziedzinowe - ile naruszeń?
[ ] 10. Karta zbioru - kto, skąd, na jakiej podstawie, do czego wolno użyć?
```

### Zapisujemy oczyszczony zbiór

Efekty pracy zapisujemy do pliku, żeby kolejny temat mógł zacząć od gotowego materiału.

In [22]:
from pathlib import Path

KATALOG_ROBOCZY = Path("wyniki")
KATALOG_ROBOCZY.mkdir(exist_ok=True)

df.to_csv(KATALOG_ROBOCZY / "t01_pracownicy_po_audycie.csv", index=False)
print("Zapisano:", KATALOG_ROBOCZY / "t01_pracownicy_po_audycie.csv")
print("Wymiary:", df.shape)

Zapisano: wyniki/t01_pracownicy_po_audycie.csv
Wymiary: (1215, 11)


---

## 9. Sprawdź się (do samodzielnego wykonania)

1. Wczytaj zbiór `titanic` i uruchom na nim funkcję `profil`. Które kolumny mają najwięcej
   braków? Czy któraś nadaje się do usunięcia w całości?
2. Zbiór `mpg` ma braki tylko w jednej kolumnie. W której i ile ich jest?
3. Napisz regułę poprawności dla `titanic`, która sprawdzi, czy kolumny `survived` i `alive`
   mówią to samo. Czy są zgodne?

In [23]:
# 1. Titanic: profil i kolumny z największą liczbą braków
titanic = wczytaj("titanic")
profil_titanic = profil(titanic)

print("Titanic — kolumny z brakami:")
display(
    profil_titanic[profil_titanic["braki"] > 0]
    .sort_values("braki", ascending=False)
)

# 2. MPG: w której kolumnie i ile jest braków?
mpg = wczytaj("mpg")

print("MPG — kolumny z brakami:")
print(mpg.isna().sum()[lambda s: s > 0])

# 3. Titanic: czy survived i alive są zgodne?
zgodne = titanic["survived"].eq(titanic["alive"].map({"no": 0, "yes": 1}))

print("Liczba niezgodnych wierszy:", (~zgodne).sum())
display(titanic.loc[~zgodne, ["survived", "alive"]].head())

Titanic — kolumny z brakami:


,typ,niepuste,braki,braki_%,unikalne,unikalne_%,przyklady,min,mediana,max
kolumna,,,,,,,,,,
deck,object,203,688,77.2,7,0.8,"C, E, G",NaN,NaN,NaN
age,float64,714,177,19.9,88,9.9,"22.0, 38.0, 26.0",0.42,28.0,80.0
embarked,object,889,2,0.2,3,0.3,"S, C, Q",NaN,NaN,NaN
embark_town,object,889,2,0.2,3,0.3,"Southampton, Cherbourg, Queenstown",NaN,NaN,NaN


MPG — kolumny z brakami:
horsepower    6
dtype: int64
Liczba niezgodnych wierszy: 0


,survived,alive
